# Pill bottle orientation classifier (T17)

Trains a **YOLOv8 classification** model that looks at a crop of the pill bottle and says **upright** or **lying** (knocked over / on its side).
On the robot, `orientation_classifier` runs it on the bottle box from `bottle_detector`, and `bottle_localizer` combines it with the depth fit.

Why classification and not a second detector: you already have a detector that finds the bottle. Classifying its crop means **no boxes to draw**: you label by clicking a button per picture.

Steps
1. Take head-camera pictures of the bottle in different orientations (on the robot):
   `python3 testing/capture_training_images.py --name upright` and `--name lying` (move the bottle between shots: different spots, distances, heights, turned around).
   Aim for **100+ per class**. Copy `~/stretch_training_images/` to Google Drive.
2. Run the cells below in order. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## Settings: change these paths to match your Drive

In [ ]:
DRIVE = "/content/drive/MyDrive/pill_bottle"
DETECTOR = f"{DRIVE}/final_pill_bottle.pt"   # your existing detector (classes pill_bottle, cap, label)
IMAGES = f"{DRIVE}/stretch_training_images"  # UPRIGHT head-camera pictures (capture_training_images.py)
CROPS = f"{DRIVE}/orientation_crops"         # crops + your sorting, kept on Drive so a runtime reset doesn't lose them
WORK = "/content/orientation"
CLASSES = ["upright", "lying"]               # add "tilted" only if you have 50+ tilted pictures
PAD = 0.10                                   # crop padding; must match orientation_classifier's pad parameter

## 1. Crop every bottle the detector finds
Crops go to `orientation_crops/unsorted/`. Pictures already cropped before are skipped, so you can add pictures later and re-run.

In [ ]:
import os, glob, shutil, cv2
from ultralytics import YOLO

det = YOLO(DETECTOR)
assert "pill_bottle" in det.names.values(), f"detector classes are {det.names}"
os.makedirs(f"{CROPS}/unsorted", exist_ok=True)
already = {os.path.basename(p) for p in glob.glob(f"{CROPS}/**/*.jpg", recursive=True)}
paths = sorted(p for ext in ("jpg", "jpeg", "png") for p in glob.glob(f"{IMAGES}/**/*.{ext}", recursive=True))
new = 0
for p in paths:
    img = cv2.imread(p)
    if img is None:
        continue
    r = det(img, conf=0.4, verbose=False)[0]
    stem = os.path.splitext(os.path.basename(p))[0]
    for i, b in enumerate(r.boxes):
        if det.names[int(b.cls)] != "pill_bottle":
            continue
        name = f"{stem}_{i}.jpg"
        if name in already:
            continue
        x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
        px, py = PAD * (x2 - x1), PAD * (y2 - y1)
        H, W = img.shape[:2]
        crop = img[int(max(0, y1 - py)):int(min(H, y2 + py)), int(max(0, x1 - px)):int(min(W, x2 + px))]
        if crop.size:
            cv2.imwrite(f"{CROPS}/unsorted/{name}", crop)
            new += 1
print(f"{len(paths)} pictures, {new} new bottle crops in {CROPS}/unsorted")

## 2. Sort the crops (click a button for each)
Each click moves the picture into `orientation_crops/<class>/`. **skip** = not a bottle / too blurry. You can stop and come back: it starts from what's left.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

for c in CLASSES + ["skip"]:
    os.makedirs(f"{CROPS}/{c}", exist_ok=True)
todo = sorted(glob.glob(f"{CROPS}/unsorted/*.jpg"))
pic = widgets.Image(format="jpg", width=256)
info = widgets.Label()
state = {"i": 0}

def show():
    if state["i"] >= len(todo):
        info.value = "All sorted! Run the next cell."
        pic.layout.display = "none"
        return
    pic.value = open(todo[state["i"]], "rb").read()
    info.value = f"{state['i'] + 1} / {len(todo)}"

def sort_into(button):
    if state["i"] >= len(todo):
        return
    p = todo[state["i"]]
    shutil.move(p, f"{CROPS}/{button.description}/{os.path.basename(p)}")
    state["i"] += 1
    show()

buttons = [widgets.Button(description=c) for c in CLASSES + ["skip"]]
for b in buttons:
    b.on_click(sort_into)
display(widgets.VBox([info, pic, widgets.HBox(buttons)]))
show()

## 3. Train / validation split
Validation = the **last 20 % of each class by capture time** (file names start with the time). Consecutive shots of the same placement look almost identical, so a random split would put near-copies in both sets and make the accuracy look better than it is.

In [ ]:
DATA = f"{WORK}/dataset"
shutil.rmtree(DATA, ignore_errors=True)
for c in CLASSES:
    files = sorted(glob.glob(f"{CROPS}/{c}/*.jpg"))
    if len(files) < 10:
        raise SystemExit(f"only {len(files)} '{c}' crops; take more pictures (100+ per class is a good target)")
    n_val = max(2, len(files) // 5)
    for split, fs in (("train", files[:-n_val]), ("val", files[-n_val:])):
        os.makedirs(f"{DATA}/{split}/{c}", exist_ok=True)
        for f in fs:
            shutil.copy(f, f"{DATA}/{split}/{c}/")
    print(f"{c:8s} train {len(files) - n_val:4d}   val {n_val:4d}")

## 4. Train

In [ ]:
model = YOLO("yolov8n-cls.pt")       # small and fast; fine on the robot's CPU
model.train(data=DATA, epochs=40, imgsz=224, batch=32, patience=10,
            project=f"{WORK}/runs", name="orientation", exist_ok=True)

## 5. Check it: accuracy and the pictures it gets wrong

In [ ]:
from IPython.display import Image as Show
best = f"{WORK}/runs/orientation/weights/best.pt"
m = YOLO(best)
print("validation top-1 accuracy:", round(m.val(data=DATA, split="val", verbose=False).top1, 3))
wrong = []
for c in CLASSES:
    for f in sorted(glob.glob(f"{DATA}/val/{c}/*.jpg")):
        r = m(f, verbose=False)[0]
        pred = m.names[int(r.probs.top1)]
        if pred != c:
            wrong.append((f, c, pred, float(r.probs.top1conf)))
print(f"{len(wrong)} wrong")
for f, c, pred, conf in wrong[:12]:
    print(f"{os.path.basename(f)}: really {c}, predicted {pred} ({conf:.2f})")
    display(Show(f, width=160))

## 6. Save to Drive, then copy to the robot
```bash
# on your laptop, after downloading orientation_cls.pt from Drive:
scp orientation_cls.pt hello-robot@192.168.1.172:~/pill_bottle_robot/
# on the robot:
ros2 launch pill_bottle_nodes nodes.launch.py stage:=orientation orientation_model:=$HOME/pill_bottle_robot/orientation_cls.pt
ros2 run pill_bottle_nodes test_orientation
```

In [ ]:
shutil.copy(best, f"{DRIVE}/orientation_cls.pt")
print("saved", f"{DRIVE}/orientation_cls.pt")

## (Optional) try one new picture

In [ ]:
from google.colab import files
for name in files.upload():
    img = cv2.imread(name)
    r = det(img, conf=0.4, verbose=False)[0]
    for b in r.boxes:
        if det.names[int(b.cls)] != "pill_bottle":
            continue
        x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
        px, py = PAD * (x2 - x1), PAD * (y2 - y1)
        H, W = img.shape[:2]
        crop = img[int(max(0, y1 - py)):int(min(H, y2 + py)), int(max(0, x1 - px)):int(min(W, x2 + px))]
        p = m(crop, verbose=False)[0]
        print(f"{name}: {m.names[int(p.probs.top1)]} ({float(p.probs.top1conf):.2f})")